# Étape 2 — Compréhension du dataset

## 1. Introduction

Le projet étudie la segmentation de clients d'un distributeur en gros à partir de leurs dépenses annuelles. Le dataset Wholesale Customers vient du UCI Machine Learning Repository, ID 292. La segmentation est un apprentissage non supervisé : les groupes ne sont pas fournis à l'avance. Une feature décrit une observation et un cluster regroupe des observations similaires. Aucun prétraitement ni clustering n'est réalisé dans cette étape.

## 2. Chargement du dataset

Le chemin est recherché depuis le dossier courant et ses parents afin de fonctionner depuis la racine du projet ou depuis le dossier notebooks.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

candidats = [Path.cwd() / 'data' / 'raw' / 'Wholesale customers data.csv']
candidats += [p / 'data' / 'raw' / 'Wholesale customers data.csv' for p in Path.cwd().parents]
csv_path = next((p.resolve() for p in candidats if p.exists()), None)
if csv_path is None:
    raise FileNotFoundError('Dataset introuvable dans data/raw/.')
df = pd.read_csv(csv_path)
print('Fichier chargé :', csv_path)
print('Dimensions :', df.shape[0], 'lignes x', df.shape[1], 'colonnes')
print('Colonnes :', list(df.columns))
df.head()

Fichier chargé : C:\Users\hp\Desktop\ML2-Customer-Segmentation\data\raw\Wholesale customers data.csv
Dimensions : 440 lignes x 8 colonnes
Colonnes : ['Channel', 'Region', 'Fresh', 'Milk', 'Grocery', 'Frozen', 'Detergents_Paper', 'Delicassen']


,Channel,Region,Fresh,Milk,Grocery,Frozen,Detergents_Paper,Delicassen
0,2,3,12669,9656,7561,214,2674,1338
1,2,3,7057,9810,9568,1762,3293,1776
2,2,3,6353,8808,7684,2405,3516,7844
3,1,3,13265,1196,4221,6404,507,1788
4,2,3,22615,5410,7198,3915,1777,5185


## 3. Description des variables

Channel et Region sont des catégories codées par des entiers. Les six dépenses sont quantitatives et seront les variables du clustering initial. Channel et Region serviront à l'interprétation.

In [2]:
depenses = ['Fresh', 'Milk', 'Grocery', 'Frozen', 'Detergents_Paper', 'Delicassen']
categoricales = ['Channel', 'Region']
descriptions = {
'Channel': ('Canal de distribution', 'catégorielle codée', 'int64', 'interprétation'),
'Region': ('Région du client', 'catégorielle codée', 'int64', 'interprétation'),
'Fresh': ('Dépenses en produits frais', 'numérique quantitative', 'int64', 'clustering initial'),
'Milk': ('Dépenses en produits laitiers', 'numérique quantitative', 'int64', 'clustering initial'),
'Grocery': ('Dépenses en épicerie', 'numérique quantitative', 'int64', 'clustering initial'),
'Frozen': ('Dépenses en produits surgelés', 'numérique quantitative', 'int64', 'clustering initial'),
'Detergents_Paper': ('Dépenses en détergents et papier', 'numérique quantitative', 'int64', 'clustering initial'),
'Delicassen': ('Dépenses en produits delicatessen', 'numérique quantitative', 'int64', 'clustering initial')}
variables = pd.DataFrame([(n, *descriptions[n]) for n in df.columns], columns=['Nom', 'Signification', 'Type statistique réel', 'Type Python détecté', 'Rôle envisagé'])
variables

,Nom,Signification,Type statistique réel,Type Python détecté,Rôle envisagé
0,Channel,Canal de distribution,catégorielle codée,int64,interprétation
1,Region,Région du client,catégorielle codée,int64,interprétation
2,Fresh,Dépenses en produits frais,numérique quantitative,int64,clustering initial
3,Milk,Dépenses en produits laitiers,numérique quantitative,int64,clustering initial
4,Grocery,Dépenses en épicerie,numérique quantitative,int64,clustering initial
5,Frozen,Dépenses en produits surgelés,numérique quantitative,int64,clustering initial
6,Detergents_Paper,Dépenses en détergents et papier,numérique quantitative,int64,clustering initial
7,Delicassen,Dépenses en produits delicatessen,numérique quantitative,int64,clustering initial


## 4. Statistiques descriptives

Le coefficient de variation est égal à écart-type / moyenne x 100. Les quartiles, l'étendue et les mesures de tendance centrale sont calculés pour les six dépenses.

In [3]:
statistiques = pd.DataFrame(index=depenses)
statistiques['Moyenne'] = df[depenses].mean()
statistiques['Médiane'] = df[depenses].median()
statistiques['Écart-type'] = df[depenses].std()
statistiques['Minimum'] = df[depenses].min()
statistiques['Q1'] = df[depenses].quantile(.25)
statistiques['Q3'] = df[depenses].quantile(.75)
statistiques['Maximum'] = df[depenses].max()
statistiques['Étendue'] = statistiques['Maximum'] - statistiques['Minimum']
statistiques['Coefficient de variation (%)'] = 100 * statistiques['Écart-type'] / statistiques['Moyenne']
statistiques.round(2)

,Moyenne,Médiane,Écart-type,Minimum,Q1,Q3,Maximum,Étendue,Coefficient de variation (%)
Fresh,12000.30,8504.0,12647.33,3,3127.75,16933.75,112151,112148,105.39
Milk,5796.27,3627.0,7380.38,55,1533.00,7190.25,73498,73443,127.33
Grocery,7951.28,4755.5,9503.16,3,2153.00,10655.75,92780,92777,119.52
Frozen,3071.93,1526.0,4854.67,25,742.25,3554.25,60869,60844,158.03
Detergents_Paper,2881.49,816.5,4767.85,3,256.75,3922.00,40827,40824,165.46
Delicassen,1524.87,965.5,2820.11,3,408.25,1820.25,47943,47940,184.94


## 5. Qualité du dataset

Les contrôles décrivent les données sans les modifier. Les valeurs extrêmes potentielles utilisent la règle IQR et ne sont pas supprimées.

In [4]:
controle = pd.Series({'Valeurs manquantes': int(df.isna().sum().sum()), 'Doublons exacts': int(df.duplicated().sum()), 'Dépenses négatives': int((df[depenses] < 0).sum().sum())})
display(controle)
display(df.dtypes.to_frame('Type Python'))
modalites = pd.DataFrame({'Valeurs observées': [sorted(df[c].unique()) for c in categoricales], 'Nombre de modalités': [df[c].nunique() for c in categoricales]}, index=categoricales)
display(modalites)
zeros = pd.DataFrame({'Nombre de zéros': (df[depenses] == 0).sum(), 'Pourcentage (%)': 100 * (df[depenses] == 0).mean()})
display(zeros.round(2))
q1 = df[depenses].quantile(.25); q3 = df[depenses].quantile(.75); iqr = q3 - q1
extremes = pd.DataFrame({'Borne inférieure': q1 - 1.5 * iqr, 'Borne supérieure': q3 + 1.5 * iqr, 'Nombre potentiel': [(((df[c] < q1[c] - 1.5 * iqr[c]) | (df[c] > q3[c] + 1.5 * iqr[c])).sum()) for c in depenses]})
display(extremes.round(2))

Valeurs manquantes    0
Doublons exacts       0
Dépenses négatives    0
dtype: int64

,Type Python
Channel,int64
Region,int64
Fresh,int64
Milk,int64
Grocery,int64
Frozen,int64
Detergents_Paper,int64
Delicassen,int64


,Valeurs observées,Nombre de modalités
Channel,"[1, 2]",2
Region,"[1, 2, 3]",3


,Nombre de zéros,Pourcentage (%)
Fresh,0,0.0
Milk,0,0.0
Grocery,0,0.0
Frozen,0,0.0
Detergents_Paper,0,0.0
Delicassen,0,0.0


,Borne inférieure,Borne supérieure,Nombre potentiel
Fresh,-17581.25,37642.75,20
Milk,-6952.88,15676.12,28
Grocery,-10601.12,23409.88,24
Frozen,-3475.75,7772.25,43
Detergents_Paper,-5241.12,9419.88,30
Delicassen,-1709.75,3938.25,27


## 6. Analyse descriptive préliminaire

Cette analyse reste tabulaire. Les histogrammes, boxplots, heatmaps et visualisations avancées sont réservés à l'étape 3.

In [5]:
channel = pd.DataFrame({'Effectif': df.Channel.value_counts().sort_index(), 'Pourcentage (%)': 100 * df.Channel.value_counts(normalize=True).sort_index()})
region = pd.DataFrame({'Effectif': df.Region.value_counts().sort_index(), 'Pourcentage (%)': 100 * df.Region.value_counts(normalize=True).sort_index()})
print('Répartition par Channel'); display(channel.round(2))
print('Répartition par Region'); display(region.round(2))
resume = statistiques[['Moyenne', 'Médiane', 'Écart-type', 'Coefficient de variation (%)']].copy()
resume['Écart relatif moyenne-médiane (%)'] = 100 * (resume['Moyenne'] - resume['Médiane']) / resume['Médiane']
variables_asymetriques = resume.index[resume['Moyenne'] > resume['Médiane']].tolist()
display(resume.round(2))
print('Variables où la moyenne dépasse la médiane :', variables_asymetriques)

Répartition par Channel


,Effectif,Pourcentage (%)
Channel,,
1,298,67.73
2,142,32.27


Répartition par Region


,Effectif,Pourcentage (%)
Region,,
1,77,17.50
2,47,10.68
3,316,71.82


,Moyenne,Médiane,Écart-type,Coefficient de variation (%),Écart relatif moyenne-médiane (%)
Fresh,12000.30,8504.0,12647.33,105.39,41.11
Milk,5796.27,3627.0,7380.38,127.33,59.81
Grocery,7951.28,4755.5,9503.16,119.52,67.20
Frozen,3071.93,1526.0,4854.67,158.03,101.31
Detergents_Paper,2881.49,816.5,4767.85,165.46,252.91
Delicassen,1524.87,965.5,2820.11,184.94,57.94


Variables où la moyenne dépasse la médiane : ['Fresh', 'Milk', 'Grocery', 'Frozen', 'Detergents_Paper', 'Delicassen']


## 7. Conclusion automatique

La synthèse ci-dessous utilise exclusivement les résultats calculés.

In [6]:
nb_extremes = int(extremes['Nombre potentiel'].sum())
qualite = 'aucune valeur manquante' if controle['Valeurs manquantes'] == 0 else str(controle['Valeurs manquantes']) + ' valeur(s) manquante(s)'
doublons = 'aucun doublon exact' if controle['Doublons exacts'] == 0 else str(controle['Doublons exacts']) + ' doublon(s)'
negatifs = 'aucune dépense négative' if controle['Dépenses négatives'] == 0 else str(controle['Dépenses négatives']) + ' dépense(s) négative(s)'
print('Structure :', df.shape[0], 'observations et', df.shape[1], 'variables.')
print('Qualité :', qualite + ',', doublons + ',', negatifs + '.')
print('Variables du clustering initial :', ', '.join(depenses) + '.')
print('Channel et Region sont réservées à l interprétation.')
print('Asymétrie potentielle : moyenne supérieure à la médiane pour', ', '.join(variables_asymetriques) if variables_asymetriques else 'aucune variable', '.')
print('La règle IQR signale', nb_extremes, 'observation(s)-variable(s) potentiellement extrême(s), sans suppression.')
print('Prétraitements à étudier plus tard : effet des échelles, asymétrie et valeurs extrêmes. Aucune transformation n est appliquée ici.')

Structure : 440 observations et 8 variables.
Qualité : aucune valeur manquante, aucun doublon exact, aucune dépense négative.
Variables du clustering initial : Fresh, Milk, Grocery, Frozen, Detergents_Paper, Delicassen.
Channel et Region sont réservées à l interprétation.
Asymétrie potentielle : moyenne supérieure à la médiane pour Fresh, Milk, Grocery, Frozen, Detergents_Paper, Delicassen .
La règle IQR signale 172 observation(s)-variable(s) potentiellement extrême(s), sans suppression.
Prétraitements à étudier plus tard : effet des échelles, asymétrie et valeurs extrêmes. Aucune transformation n est appliquée ici.
